# 02 — Cleaning QA & Data Quality Validation

This notebook follows Notebook 01 and performs **structured QA checks** on the cleaned dataset.

It does not change business logic. It documents quality, nulls, ranges, and saves an optional CLI mirror file for reproducibility comparison.

---
## 1) Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_olist_dataset.csv'
CLI_MIRROR_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_olist_dataset_cli.csv'

df = pd.read_csv(DATA_PATH)
print(f'Loaded cleaned dataset: {len(df):,} rows × {len(df.columns)} columns')

---
## 2) Schema and Type Health

In [ ]:
dtype_view = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(t) for t in df.dtypes],
    'non_null': [df[c].notna().sum() for c in df.columns],
    'nulls': [df[c].isna().sum() for c in df.columns],
    'null_pct': [df[c].isna().mean() * 100 for c in df.columns],
}).sort_values('null_pct', ascending=False)
dtype_view.head(20)

---
## 3) Key Integrity Checks

In [ ]:
checks = {}
checks['row_count'] = len(df)
checks['col_count'] = len(df.columns)
checks['unique_orders'] = df['order_id'].nunique()
checks['delivered_only_ratio'] = (df['order_status'].eq('delivered').mean() * 100) if 'order_status' in df.columns else np.nan
checks['review_score_missing_pct'] = df['review_score'].isna().mean() * 100
checks['delivery_delay_missing_pct'] = df['delivery_delay_days'].isna().mean() * 100
checks['on_time_rate_pct'] = df['is_on_time'].astype(float).mean() * 100

pd.Series(checks)

---
## 4) Numeric Sanity Bands (Spot gross outliers / impossible values)

In [ ]:
numeric_cols = [c for c in [
    'price', 'freight_value', 'total_item_value', 'total_payment_value',
    'delivery_delay_days', 'actual_delivery_days', 'approval_lag_hours'
] if c in df.columns]

df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

---
## 5) Save Optional CLI Mirror Copy

This keeps compatibility with existing repo artifacts used elsewhere.

In [ ]:
df.to_csv(CLI_MIRROR_PATH, index=False)
print('Saved:', CLI_MIRROR_PATH)

---
## Notebook Output

- Input: `data/processed/cleaned_olist_dataset.csv`
- Optional mirror output: `data/processed/cleaned_olist_dataset_cli.csv`

Next notebook: **`03_eda.ipynb`**